# Griego: revisión de 30 pares y submuestra fija de 300k (issue 007)

Corre en Google Colab **sin GPU** (basta el entorno de CPU). Hace, en orden:

1. Monta la carpeta compartida de Drive y clona el repositorio.
2. Copia `artefactos/datos/en_el` a la máquina de Colab y verifica su huella (la del issue 001).
3. **Revisión:** muestra 30 pares aleatorios del entrenamiento (semilla fija) y guarda en Drive una tabla Markdown para calificarlos (correcto / parcial / incorrecto).
4. **Submuestra** (solo si el equipo decide seguir con griego): genera `artefactos/datos/en_el_300k/` con 300 000 pares de `train` elegidos con semilla fija y `validation`/`test` **completas**, la sube a Drive y comprueba que el MD5 en Drive coincide con el local.

**Uso:** editar solo la celda de configuración y *Entorno de ejecución → Ejecutar todas*. La primera vez, con `CONTINUAR_CON_GRIEGO = False` (solo revisión). Tras la decisión del equipo, cambiarlo a `True` y volver a ejecutar.

La lógica está en `traduccion/datos.py` (`pares_para_revision`, `tabla_revision`, `guardar_submuestra`) y se prueba en `tests/test_griego.py`.

## 0. Configuración

In [ ]:
# Carpeta compartida de Drive (la misma que en ejecutor_colab.ipynb): se localiza por su ID
# a través del acceso directo en "Mi unidad". CARPETA_DRIVE solo si la resolución por ID falla.
ID_CARPETA_DRIVE = "17TOnMl178b--icGi1UzQdmjtxNqDO5dm"
CARPETA_DRIVE = None  # p. ej. "/content/drive/MyDrive/<nombre de la carpeta>"

URL_REPO = "https://github.com/FROY33/Sistema-de-traduccion-automatica"
COMMIT = "origin/main"  # o un hash concreto; el hash completo usado queda en el resumen

# False: solo revisión. True: además genera y sube la submuestra (tras la decisión del equipo).
CONTINUAR_CON_GRIEGO = False

INSTALAR_DEPENDENCIAS = True

# Huella de artefactos/datos/en_el anotada en el issue 001.
HUELLA_EN_EL = {"md5_datos": "963ba2e185e9c82dbde68c59ff4f7ceb", "bytes_datos": 1515343376}

## 1. Montar Drive y clonar el repositorio

In [ ]:
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
if CARPETA_DRIVE is None:
    por_id = Path("/content/drive/.shortcut-targets-by-id") / ID_CARPETA_DRIVE
    if not por_id.is_dir():
        raise FileNotFoundError(
            f"No se encuentra la carpeta compartida (ID {ID_CARPETA_DRIVE}). Añade un acceso directo a ella "
            "en tu Mi unidad (clic derecho → Organizar → Añadir acceso directo) o fija CARPETA_DRIVE."
        )
    subcarpetas = [p for p in por_id.iterdir() if p.is_dir()]
    CARPETA_DRIVE = str(subcarpetas[0] if len(subcarpetas) == 1 else por_id)
DRIVE = Path(CARPETA_DRIVE)
ARTEFACTOS = DRIVE / "artefactos"
print(f"Drive: {DRIVE}")

CARPETA_REPO = Path("/content/Sistema-de-traduccion-automatica")
if not (CARPETA_REPO / ".git").exists():
    subprocess.run(["git", "clone", "--quiet", URL_REPO, str(CARPETA_REPO)], check=True)
else:
    subprocess.run(["git", "fetch", "--quiet", "origin"], cwd=CARPETA_REPO, check=True)
subprocess.run(["git", "checkout", "--quiet", "--detach", COMMIT], cwd=CARPETA_REPO, check=True)
COMMIT_COMPLETO = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=CARPETA_REPO, capture_output=True, text=True, check=True
).stdout.strip()
print(f"Repositorio en {COMMIT_COMPLETO}")

if str(CARPETA_REPO) not in sys.path:
    sys.path.insert(0, str(CARPETA_REPO))
from traduccion import ejecutor  # solo biblioteca estándar: se puede importar antes de instalar

if INSTALAR_DEPENDENCIAS:
    texto = (CARPETA_REPO / "requirements.txt").read_text(encoding="utf-8")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *ejecutor.dependencias_para_colab(texto)], check=True)

from traduccion import datos

## 2. Copiar `datos/en_el` a Colab y verificar su huella

Leer pares sueltos directamente desde Drive es muy lento; se copia la carpeta (≈1.5 GB) a `/content` una vez y se verifica que es la del issue 001. Si la huella no coincide, se detiene.

In [ ]:
import shutil

ORIGEN_DRIVE = ARTEFACTOS / "datos" / "en_el"
ORIGEN = Path("/content/en_el")
if not ORIGEN.exists():
    print(f"Copiando {ORIGEN_DRIVE} → {ORIGEN} …")
    shutil.copytree(ORIGEN_DRIVE, ORIGEN)

huella = {"md5_datos": ejecutor.md5_carpeta(ORIGEN), "bytes_datos": ejecutor.bytes_carpeta(ORIGEN)}
print(huella)
if huella != HUELLA_EN_EL:
    raise RuntimeError(f"La carpeta en_el no coincide con la del issue 001: {huella} ≠ {HUELLA_EN_EL}")
print("Huella de en_el verificada.")

## 3. Revisión manual de 30 pares

Se eligen con `indices_muestra` (MD5 de `semilla:índice`), así que son los mismos en cualquier máquina. El texto está normalizado y en minúsculas, como en el entrenamiento. La tabla se guarda en Drive en `revision/revision_griego_007.md`.

Al calificar, fijarse en:
- **Alineación:** ¿el griego traduce la frase inglesa? `correcto`, `parcial` (falta o sobra un trozo, paráfrasis libre) o `incorrecto` (otra frase).
- **Verso frente a prosa**, **ruido** (números de verso, notas, referencias, caracteres raros) y pares muy cortos o muy desiguales en longitud.

In [ ]:
from datasets import load_from_disk
from IPython.display import Markdown, display

TRAIN = load_from_disk(str(ORIGEN / "train"))
print(f"Pares de entrenamiento en en_el: {len(TRAIN):,}")

PARES_REVISION = datos.pares_para_revision(TRAIN)
for numero, par in enumerate(PARES_REVISION, start=1):
    print(f"── {numero:2d} · índice {par['indice']} · tokens en/el {par['tokens_ingles']}/{par['tokens_clasico']}")
    print(f"   EN: {par['ingles']}")
    print(f"   EL: {par['clasico']}\n")

TABLA = datos.tabla_revision(PARES_REVISION)
RUTA_TABLA = DRIVE / "revision" / "revision_griego_007.md"
RUTA_TABLA.parent.mkdir(parents=True, exist_ok=True)
if RUTA_TABLA.exists():
    print(f"{RUTA_TABLA} ya existe: no se sobrescribe (puede tener calificaciones).")
else:
    RUTA_TABLA.write_text(
        f"# Revisión de pares griegos (issue 007)\n\nCommit {COMMIT_COMPLETO}, semilla "
        f"{datos.SEMILLA_REVISION_GRIEGO}, {len(TRAIN):,} pares en train.\n\n" + TABLA,
        encoding="utf-8",
    )
    print(f"Tabla para calificar guardada en {RUTA_TABLA}")
display(Markdown(TABLA))

### Qué hacer después de la revisión

1. Llenar las columnas *Valoración* y *Nota* de `revision/revision_griego_007.md` (en Drive, o copiándola al issue 007).
2. Contar correctos / parciales / incorrectos y elegir 3–4 ejemplos de problemas.
3. Con el equipo, decidir si se sigue con griego y anotarlo en el issue 007, junto con 1–2 frases para la sección de limitaciones del artículo.
4. Si se sigue: `CONTINUAR_CON_GRIEGO = True` en la configuración y volver a ejecutar todo.

## 4. Submuestra fija de 300 000 pares

Solo `train` se submuestrea (semilla `0`); `validation` y `test` se copian completas. Se genera primero en `/content`, se sube a `artefactos/datos/en_el_300k/` y se comprueba que el MD5 en Drive es el mismo que el local. Si la carpeta ya existe en Drive, **no se rehace**: solo se recalcula y muestra su huella.

Los modelos griegos (issue 012) la usan con `evaluar.py --par en-el --datos <Drive>/artefactos/datos/en_el_300k` y verifican este MD5.

In [ ]:
import json

DESTINO_DRIVE = ARTEFACTOS / datos.CARPETA_SUBMUESTRA_GRIEGO
RUTA_RESUMEN = ARTEFACTOS / (datos.CARPETA_SUBMUESTRA_GRIEGO + ".json")

if not CONTINUAR_CON_GRIEGO:
    print("CONTINUAR_CON_GRIEGO = False: no se genera la submuestra.")
elif DESTINO_DRIVE.exists():
    print(f"{DESTINO_DRIVE} ya existe; no se rehace. Huella actual:")
    print({"md5_datos": ejecutor.md5_carpeta(DESTINO_DRIVE), "bytes_datos": ejecutor.bytes_carpeta(DESTINO_DRIVE)})
    if RUTA_RESUMEN.exists():
        print(RUTA_RESUMEN.read_text(encoding="utf-8"))
else:
    LOCAL = Path("/content/en_el_300k")
    if LOCAL.exists():
        shutil.rmtree(LOCAL)
    RESUMEN = datos.guardar_submuestra(ORIGEN, LOCAL)
    RESUMEN["commit"] = COMMIT_COMPLETO
    print(json.dumps(RESUMEN, indent=2))

    print(f"Subiendo a {DESTINO_DRIVE} …")
    shutil.copytree(LOCAL, DESTINO_DRIVE)
    md5_drive = ejecutor.md5_carpeta(DESTINO_DRIVE)
    if md5_drive != RESUMEN["md5_datos"]:
        raise RuntimeError(f"El MD5 en Drive ({md5_drive}) no coincide con el local ({RESUMEN['md5_datos']}).")
    RUTA_RESUMEN.write_text(json.dumps(RESUMEN, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    print("Submuestra subida y verificada en Drive. Fila para el issue 007:\n")
    print("| Carpeta | Semilla | Pares train | `md5_indices` | `md5_datos` | `bytes_datos` |")
    print("|---|---|---|---|---|---|")
    print(
        f"| `{datos.CARPETA_SUBMUESTRA_GRIEGO}` | {RESUMEN['semilla']} | {RESUMEN['tamano']:,} de {RESUMEN['n_train_origen']:,} "
        f"| `{RESUMEN['md5_indices']}` | `{RESUMEN['md5_datos']}` | {RESUMEN['bytes_datos']:,} |"
    )